# Locality check: position-shuffling test

The plain 1D CNN on the whole pan-genome, with and without shuffled variant order (the paper's locality argument for the Motif Pattern Encoder). Independent of the rest.

**Before you start**

1. Settings → Accelerator → **GPU T4 x2**. Settings → Internet → **On**.
2. Add Input → your dataset **trilens-data**. (The code is cloned from GitHub.)
3. Add-ons → Secrets → **GITHUB_TOKEN** → switch it on for this notebook.
4. **Save Version → Save & Run All (Commit)**. You can close the tab.

Output: `shuffle_test_outputs.zip`. After every step it is sent to the GitHub release `outputs-shuffle_test`, and its text results to the branch `results/shuffle_test`. If the session stops (12-hour limit), just run it again: it downloads its own release and skips the finished steps.

In [ ]:
# 1. Set up: code, data, and the outputs of the earlier notebooks
GITHUB_REPO = "FaiazPronoy/TriLens-AMR"   # where the code lives and the outputs go
CODE_REF    = "main"                       # branch, tag or commit to run
RESUME_OWN  = True                       # continue from this notebook's own release

NAME   = "shuffle_test"
LENSB  = "fixed"          # "fixed" = delivered model, "standard" = ablation model
WRITES = ["pillar1"]
NEEDS  = []
NEEDS_GPU = True

import os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

INPUT   = Path("/kaggle/input")
BASE    = Path("/tmp/trilens")           # outside /kaggle/working: only the zip is saved as output
WORK    = BASE / "code"
RES     = WORK / "results"
OUT_ZIP = Path(f"/kaggle/working/{NAME}_outputs.zip")
PREFIX  = "standard/" if LENSB == "standard" else ""
STANDARD_STAGES = ("standard_random", "standard_splits")
FIXED_TRILENS   = ("trilens_random", "trilens_splits")

def find_one(name):
    hits = sorted(INPUT.rglob(name))
    return hits[0] if hits else None

def _secret(key):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        return os.environ.get(key)

TOKEN = _secret("GITHUB_TOKEN")      # Add-ons -> Secrets -> GITHUB_TOKEN, switched on for this notebook
print("github :", GITHUB_REPO, "(token found)" if TOKEN else "(no token: nothing is uploaded)")

# code: the dataset trilens-code if attached, otherwise a clone of the repository
entry = find_one("run_experiment.py")
if entry is not None:
    code_src, CODE_VERSION = entry.parent.parent, "dataset trilens-code"
else:
    repo_dir = BASE / "repo"
    if not repo_dir.exists():
        url = f"https://github.com/{GITHUB_REPO}.git"
        auth = []        # the token goes in a header for this one command, never into .git/config
        if TOKEN:
            import base64
            auth = ["-c", "http.extraheader=AUTHORIZATION: basic "
                    + base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()]
        r = subprocess.run(["git", *auth, "clone", "--quiet", url, str(repo_dir)], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git clone failed: " + (r.stderr or "").replace(TOKEN or "\0", "***"))
        subprocess.run(["git", "checkout", "--quiet", CODE_REF], cwd=repo_dir, check=True)
    code_src = repo_dir / "code"
    CODE_VERSION = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo_dir,
                                  capture_output=True, text=True).stdout.strip()
print("version:", CODE_VERSION)
if not WORK.exists():
    def _skip(d, names):
        top = {"data", "results", "experiments", "outputs"} if Path(d) == code_src else set()
        return [n for n in names if n in top or n == "__pycache__"]
    shutil.copytree(code_src, WORK, ignore=_skip)
print("code   :", code_src)

# data (dataset trilens-data): symlinks, never copies
master = find_one("arcadia_v2_master_features.pkl")
if master is None:
    raise FileNotFoundError("add the dataset trilens-data (Add Input)")
proc, giessen = WORK / "data" / "processed", WORK / "data" / "Giessen_dataset"
proc.mkdir(parents=True, exist_ok=True); giessen.mkdir(parents=True, exist_ok=True)
for f in sorted(master.parent.iterdir()):
    if f.is_file():
        dst = (giessen if f.name.startswith("cip_ctx_ctz_gen") else proc) / f.name
        if not dst.exists():
            dst.symlink_to(f)
for n in ("cip_ctx_ctz_gen_multi_data.csv", "cip_ctx_ctz_gen_pheno.csv"):
    f = find_one(n)
    if f is not None and not (giessen / n).exists():
        (giessen / n).symlink_to(f)
print("data   :", master.parent, f"({len(list(proc.iterdir()))} files)")

# The random-split gene-filtered file has 42 columns with values outside the SNP codes
# 0..3 (found by data_check). Re-read those columns from the SNP matrix; the script
# refuses if anything else in the file differs from the matrix. Report: results/diagnostics/.
gfp = proc / "arcadia_v2_gene_filtered.pkl"
fixed = BASE / "fixed" / gfp.name
DATA_STATE = "repaired" if (gfp.is_symlink() and Path(os.readlink(gfp)) == fixed) else "unchecked"
if gfp.is_symlink() and Path(os.readlink(gfp)) != fixed:
    r = subprocess.run([sys.executable, str(WORK / "scripts" / "repair_gene_filtered.py"),
                        "--src", str(gfp.resolve()), "--snp", str((proc / "arcadia_v2_snp_matrix.pkl").resolve()),
                        "--out", str(fixed), "--report", str(WORK / "results" / "diagnostics" / "data_repair.json")],
                       capture_output=True, text=True)
    print("\n".join(l for l in r.stdout.splitlines() if '"status"' in l or "columns" in l))
    if r.returncode == 0:
        gfp.unlink(); gfp.symlink_to(fixed)
        DATA_STATE = "repaired"
        print("data   : arcadia_v2_gene_filtered.pkl repaired ->", fixed)
    elif r.returncode == 2:
        DATA_STATE = "clean"
    else:
        raise RuntimeError("repair_gene_filtered refused the file - read the report above:\n" + r.stdout[-2000:] + r.stderr[-2000:])

# outputs of earlier notebooks: every *_outputs.zip under /kaggle/input
# (a notebook output, or a dataset you uploaded it to). The standard-model notebooks
# keep their files under results/standard/; inside those notebooks they are unpacked
# to results/ itself, because there the standard model IS the main run.
RES.mkdir(parents=True, exist_ok=True)

def _target(stage, member):
    if LENSB == "standard":
        if stage in FIXED_TRILENS:
            return None                       # never mix the two models
        if member.startswith("standard/"):
            member = member[len("standard/"):]
    return RES / member

# Runs that read the random-split gene-filtered file. Their outputs from a run made
# before the repair (no _DATA_REPAIRED.txt in the zip) are thrown away and recomputed.
USES_RANDOM_GF = {"stage3_cascadestack_v9", "tabnet_baseline", "pillar_tabnet_phylogeny", "pillar_tabnet_mlst",
                  "pillar3_xgb_importance", "pillar3_shap", "pillar_shap_t6", "trilens_attnfix",
                  "pillar2_layer_ablation", "pillar_dedup_eval", "pillar_metrics_full"}
_pre_repair = set()
_unfinished = set()      # stages whose output has no _COMPLETE.txt (still running, or stopped)
_pa_old = set()          # stages whose output was made with the old presence/absence matrix

def _note(stage, names):
    if "_DATA_REPAIRED.txt" not in names:
        _pre_repair.add(stage)
    if "_COMPLETE.txt" not in names:
        _unfinished.add(stage)
    if "_PA_FIXED.txt" not in names:
        _pa_old.add(stage)

def _take(stage, rel, read):
    t = _target(stage, rel)
    if t is None or rel.split("/")[0].startswith("_"):
        return 0
    if stage == NAME and t.exists():
        return 0                          # a newer copy from an earlier stage is already here
    if stage in _pre_repair and rel.removeprefix("standard/").split("/")[0] in USES_RANDOM_GF:
        return 0
    t.parent.mkdir(parents=True, exist_ok=True)
    t.write_bytes(read())
    return 1

for z in sorted(INPUT.rglob("*_outputs.zip")):
    stage = z.name[: -len("_outputs.zip")]
    with zipfile.ZipFile(z) as zf:
        _note(stage, zf.namelist())
        n = sum(_take(stage, m.filename, lambda m=m: zf.read(m))
                for m in zf.infolist() if not m.is_dir())
    print(f"earlier: {z.name:32s} {n} files")
for marker in sorted(INPUT.rglob("_STAGE_*.txt")):      # an output Kaggle already unpacked
    stage, top = marker.stem[len("_STAGE_"):], marker.parent
    _note(stage, [f.name for f in top.iterdir()])
    n = sum(_take(stage, f.relative_to(top).as_posix(), f.read_bytes)
            for f in top.rglob("*") if f.is_file())
    print(f"earlier: {stage:32s} {n} files (unpacked)")

# earlier stages not attached as inputs: download them from the GitHub releases.
# Also this notebook's own release, so a stopped run continues where it stopped. It goes
# first: a newer output of an earlier stage must overwrite the copy kept in it.
# (analysis does not resume: everything it computes is cheap and derived from upstream.)
sys.path.insert(0, str(WORK / "scripts"))
import github_sync as gh
have = {z.name[: -len("_outputs.zip")] for z in INPUT.rglob("*_outputs.zip")}
have |= {m.stem[len("_STAGE_"):] for m in INPUT.rglob("_STAGE_*.txt")}
own = [NAME] if globals().get("RESUME_OWN", True) else []
for stage in [s for s in own + NEEDS if s not in have]:
    z = gh.download_asset(GITHUB_REPO, TOKEN, f"outputs-{stage}", f"{stage}_outputs.zip", BASE / "inputs")
    if z is None:
        if stage != NAME:
            print(f"earlier: {stage} NOT FOUND - run that notebook first (or attach its output)")
            _unfinished.add(stage)
        continue
    with zipfile.ZipFile(z) as zf:
        _note(stage, zf.namelist())
        n = sum(_take(stage, m.filename, lambda m=m: zf.read(m))
                for m in zf.infolist() if not m.is_dir())
    print(f"earlier: {stage:32s} {n} files (GitHub release)"
          + ("  [made before the data repair: runs on the random-split file are redone]" if stage in _pre_repair else ""))

# An earlier notebook that is still running (or stopped) has no _COMPLETE.txt yet.
# Starting now would build on its partial output, so stop here instead.
waiting = sorted(set(NEEDS) & _unfinished)
if waiting:
    raise RuntimeError(f"not finished yet (or not found): {waiting}. Run this notebook after "
                       + ", ".join(waiting) + " says 'All steps finished'.")

# The gene presence/absence matrix in the dataset marks the minority state of each locus,
# not presence (KNOWN_ISSUES, issue 12). Notebooks that read it use the copy rebuilt by pa_fix.
PA_STAGES = ("trilens_random", "trilens_splits", "standard_random", "standard_splits")
PA_STATE = "original"
if "pa_fix" in NEEDS:
    pa_fixed = RES / "pa_fix" / "arcadia_v2_gene_presence_absence.pkl"
    if not pa_fixed.exists():
        raise RuntimeError("pa_fix finished without the rebuilt matrix: read its pa_repair.json")
    pap = proc / pa_fixed.name
    if pap.exists() or pap.is_symlink():
        pap.unlink()
    pap.symlink_to(pa_fixed)
    PA_STATE = "fixed"
    print("data   : gene presence/absence -> rebuilt copy from pa_fix")
old_pa = sorted(set(NEEDS) & set(PA_STAGES) & _pa_old)
if old_pa:
    raise RuntimeError(f"{old_pa} ran on the old presence/absence matrix: run them again first.")

os.chdir(WORK)
os.environ.update(TRILENS_ROOT=str(WORK), TRILENS_MAIN_RUN="trilens_attnfix", TRILENS_LENSB=LENSB,
                  WANDB_MODE="disabled", PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch
ENV_TEXT = "\n".join([
    f"python {sys.version.split()[0]}",
    f"torch {torch.__version__} cuda {torch.version.cuda}",
    f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}",
    "", subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout])
print("\nGPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Lens B :", LENSB)
if NEEDS_GPU and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")

In [ ]:
# 2. Helpers: run a step (skipped if its output exists), save the zip after every step
FIG_DIR = BASE / "thesis" / "images"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def _keep(f):
    parts = set(f.parts)
    return not (parts & {"cache", "ckpts", "__pycache__"} or f.suffix == ".ckpt")

def package(complete=False):
    tmp = OUT_ZIP.with_suffix(".tmp")
    dirs = ([d for d in RES.iterdir() if d.is_dir() and d.name != "pa_fix"] if WRITES == ["*"]
            else [RES / d for d in WRITES])
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr(f"_STAGE_{NAME}.txt", NAME)
        z.writestr("_CODE_VERSION.txt", f"{NAME}\n{GITHUB_REPO}\n{CODE_VERSION}\n")
        if DATA_STATE in ("repaired", "clean"):
            z.writestr("_DATA_REPAIRED.txt", DATA_STATE)
        if PA_STATE == "fixed" or NAME == "pa_fix":
            z.writestr("_PA_FIXED.txt", "presence/absence matrix rebuilt by pa_fix")
        z.writestr("_ENV.txt", ENV_TEXT)            # exact package versions and GPU of this run
        if complete:                                # the notebook reached its last cell
            z.writestr("_COMPLETE.txt", "failed steps:\n" + "\n".join(FAILED))
        for d in dirs:
            for f in d.rglob("*") if d.exists() else []:
                if f.is_file() and _keep(f.relative_to(RES)):
                    z.write(f, PREFIX + f.relative_to(RES).as_posix())
        for f in sorted(FIG_DIR.glob("*.pdf")):
            z.write(f, "figures/" + f.name)
    tmp.replace(OUT_ZIP)
    print(f"[saved] {OUT_ZIP.name}  {OUT_ZIP.stat().st_size / 1e6:.1f} MB")

_last_sync = [0.0]

def sync(force=False):
    """Send the zip to the release and the text results to their branch (at most every 10 min)."""
    if not TOKEN or (not force and time.time() - _last_sync[0] < 600):
        return
    gh.upload_asset(GITHUB_REPO, TOKEN, f"outputs-{NAME}", OUT_ZIP)
    gh.push_text_results(gh.remote(GITHUB_REPO), f"results/{NAME}", RES,
                         message=f"{NAME} outputs, code {CODE_VERSION}", token=TOKEN)
    _last_sync[0] = time.time()

FAILED = []

def run(cmd, marker=None, cwd=None, env=None):
    root = Path(cwd) if cwd else WORK
    if marker and (root / "results" / marker).exists():
        print(f"[skip] {cmd}\n       (results/{marker} already exists)")
        return
    print(f"\n$ {cmd}\n", flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env={**os.environ, **(env or {})})
    last_save = time.time()
    for line in p.stdout:
        print(line, end="")
        if time.time() - last_save > 1200:        # a long step: save what it has so far
            package()                              # (finished folds, cached runs) every 20 min,
            sync()                                 # so a session that ends at 12 h loses little
            last_save = time.time()
    p.wait()
    print(f"\n[{'ok' if p.returncode == 0 else 'FAILED'}] {cmd}  ({(time.time() - t) / 60:.1f} min)")
    package()
    sync()
    if p.returncode != 0:
        FAILED.append(cmd)      # keep going: later steps that do not need this one still run

## 3. 1D CNN, giessen orig

In [ ]:
run("EXP3_ONLY=giessen_CIP_orig python -m src.run_experiment experiment=pillar1_exp3",
    "pillar1/exp3/giessen_CIP_orig_run.json")

## 4. 1D CNN, giessen shuf

In [ ]:
run("EXP3_ONLY=giessen_CIP_shuf python -m src.run_experiment experiment=pillar1_exp3",
    "pillar1/exp3/giessen_CIP_shuf_run.json")

## 5. 1D CNN, arcadia orig

In [ ]:
run("EXP3_ONLY=arcadia_CIP_orig python -m src.run_experiment experiment=pillar1_exp3",
    "pillar1/exp3/arcadia_CIP_orig_run.json")

## 6. 1D CNN, arcadia shuf

In [ ]:
run("EXP3_ONLY=arcadia_CIP_shuf python -m src.run_experiment experiment=pillar1_exp3",
    "pillar1/exp3/arcadia_CIP_shuf_run.json")

## 7. Ordered against shuffled: summary

In [ ]:
run("python -m src.run_experiment experiment=pillar1_exp3",
    "pillar1/exp3/exp3_results.csv")

## 8. Check

In [ ]:
# 8. Check: every paper number this run has produced so far, old against new
run("python scripts/diagnostics/collect_paper_numbers.py")
import pandas as pd
t = pd.read_csv(RES / "diagnostics" / "paper_numbers.csv")
print(t[t.status != "MISSING"].drop(columns="source").to_string(index=False))

In [ ]:
# 9. Save (and send to GitHub)
package(complete=True)
sync(force=True)
print("Download shuffle_test_outputs.zip from the Output tab (right side, or Data -> Output).")
if FAILED:
    print("\nThese steps FAILED (scroll up to their output):")
    for c in FAILED:
        print("  ", c)
    raise RuntimeError(f"{len(FAILED)} step(s) failed")
print("\nAll steps finished.")